# Lab 2: Policy Gradient Methods — SOLUTION

## TDDE78 — Deep Reinforcement Learning
### Linköping University, Spring 2026

---

**This notebook contains the complete solution. Do not share with students.**

In [115]:
import os
import sys
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import Video, display
import warnings
warnings.filterwarnings('ignore')

# Import lab utilities
from networks import DiscreteActorCritic
from utils import compute_returns, compute_gae, RolloutBuffer

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Resolve experiments directory relative to this notebook
_here = globals().get('__vsc_ipynb_file__', os.path.abspath(''))
_nb_dir = os.path.dirname(_here) if os.path.isfile(_here) else _here
EXPERIMENTS_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'experiments'))
print(f'Experiments directory: {EXPERIMENTS_DIR}')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(42)
print('Setup complete!')

Using device: cuda
Experiments directory: /home/amath/Desktop/course/tdde78lab/labs/lab2_policy_gradient/experiments
Setup complete!


## Explore the Environments

In [116]:
# CartPole — discrete
env_cp = gym.make('CartPole-v1')
print('=== CartPole-v1 ===')
print(f'  Observation space: {env_cp.observation_space}  shape={env_cp.observation_space.shape}')
print(f'  Action space:      {env_cp.action_space}')
obs, _ = env_cp.reset(seed=42)
print(f'  Sample obs: {obs}')
env_cp.close()

=== CartPole-v1 ===
  Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)  shape=(4,)
  Action space:      Discrete(2)
  Sample obs: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]


## Network Tests

In [117]:
# Test DiscreteActorCritic
net_d = DiscreteActorCritic(state_dim=4, action_dim=2).to(device)
test_state = torch.randn(8, 4).to(device)
action, log_prob, entropy, value = net_d.get_action(test_state)
print(f'DiscreteActorCritic:')
print(f'  action shape:   {action.shape}   dtype={action.dtype}')
print(f'  log_prob shape: {log_prob.shape}')
print(f'  entropy shape:  {entropy.shape}')
print(f'  value shape:    {value.shape}')
print(f'  params: {sum(p.numel() for p in net_d.parameters()):,}')
print('Network test passed!')

DiscreteActorCritic:
  action shape:   torch.Size([8])   dtype=torch.int64
  log_prob shape: torch.Size([8])
  entropy shape:  torch.Size([8])
  value shape:    torch.Size([8, 1])
  params: 4,675
Network test passed!


---

# Part A — Implementation

---

## REINFORCE Agent — Complete Solution

In [118]:
class REINFORCEAgent:
    """
    REINFORCE agent with optional learned baseline.

    Implements Williams (1992) REINFORCE with a shared actor-critic network.
    The value head serves as the baseline to reduce gradient variance.

    Args:
        state_dim    (int):   Dimension of state space.
        action_dim   (int):   Number of discrete actions.
        lr           (float): Learning rate.
        gamma        (float): Discount factor.
        use_baseline (bool):  If True, subtract V(s_t) from returns as baseline.
        value_coef   (float): Weight for value loss when using baseline.
        seed         (int):   Random seed.
    """

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 use_baseline=True, value_coef=0.5, seed=42):
        self.gamma = gamma
        self.use_baseline = use_baseline
        self.value_coef = value_coef

        self.network = DiscreteActorCritic(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)

    def select_action(self, state):
        """Sample an action stochastically (no gradient)."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action, log_prob, _, _ = self.network.get_action(state_t)
        return action.item(), log_prob.item()

    def update(self, states, actions, rewards):
        """
        Perform one gradient update on a complete episode.

        Args:
            states  (list): States visited during the episode.
            actions (list): Actions taken.
            rewards (list): Rewards received.

        Returns:
            pg_loss    (float): Policy gradient loss.
            value_loss (float): Value function loss (0 if no baseline).
        """
        returns = compute_returns(rewards, self.gamma)
        returns_t = torch.FloatTensor(returns).to(device)

        # Normalize returns for value function training stability
        returns_norm = returns_t
        if len(returns_t) > 1:
            returns_norm = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        states_t  = torch.FloatTensor(np.array(states)).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        _, log_probs, _, values = self.network.get_action(states_t, actions_t)
        values = values.squeeze(-1)

        if self.use_baseline:
            # Advantage = raw return - baseline; normalize for gradient stability
            advantages = returns_t - values.detach()
            if len(advantages) > 1:
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            value_loss = F.mse_loss(values, returns_norm)
        else:
            # No baseline: use normalized returns directly as advantage signal
            advantages = returns_norm
            value_loss = torch.tensor(0.0, device=device)

        pg_loss = -(log_probs * advantages).mean()
        loss = pg_loss + self.value_coef * value_loss

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.network.parameters(), 0.5)
        self.optimizer.step()

        return pg_loss.item(), value_loss.item()

print('REINFORCEAgent defined (SOLUTION)')

REINFORCEAgent defined (SOLUTION)


## PPO Agent — Complete Solution

In [ ]:
class PPOAgent:
    """
    PPO agent for discrete action spaces (CartPole-v1).

    Implements PPO-Clip (Schulman et al., 2017) with:
    - Clipped surrogate objective
    - GAE advantage estimation
    - Advantage normalization per mini-batch
    - Value loss clipping (prevents V from drifting per epoch)
    - Entropy bonus for exploration
    - Gradient norm clipping
    - Env state carried between rollouts (CleanRL-style)

    Args:
        state_dim     (int):   Dimension of state space.
        action_dim    (int):   Number of discrete actions.
        lr            (float): Learning rate.
        gamma         (float): Discount factor.
        gae_lambda    (float): GAE lambda (0=TD, 1=MC).
        clip_eps      (float): PPO clipping parameter epsilon.
        n_epochs      (int):   Number of update epochs per rollout.
        batch_size    (int):   Mini-batch size for each epoch.
        value_coef    (float): Weight for value loss.
        entropy_coef  (float): Weight for entropy bonus.
        n_steps       (int):   Environment steps collected per rollout.
        max_grad_norm (float): Gradient clipping threshold.
        seed          (int):   Random seed.
    """

    def __init__(self, state_dim, action_dim,
                 lr=2.5e-4, gamma=0.99, gae_lambda=0.95, clip_eps=0.2,
                 n_epochs=4, batch_size=64, value_coef=0.5, entropy_coef=0.01,
                 n_steps=512, max_grad_norm=0.5, seed=42):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_eps = clip_eps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.n_steps = n_steps
        self.max_grad_norm = max_grad_norm

        self.buffer = RolloutBuffer(n_steps, state_dim, action_dim, is_continuous=False)
        self.network = DiscreteActorCritic(state_dim, action_dim).to(device)

        # eps=1e-5 is standard in CleanRL PPO
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr, eps=1e-5)

        # Env state carried between rollouts — initialized by train_ppo
        self.obs = None

    def select_action(self, state):
        """Sample an action from the current policy (no gradient)."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action, log_prob, _, value = self.network.get_action(state_t)
        return action.cpu().numpy().squeeze(0), log_prob.item(), value.item()

    def collect_rollout(self, env):
        """
        Collect n_steps transitions and compute GAE.

        Uses self.obs to carry env state between rollouts (CleanRL-style),
        so rollouts can start mid-episode rather than always at episode boundaries.

        Args:
            env: A Gymnasium environment instance.

        Returns:
            float: Mean episode reward over completed episodes in this rollout.
        """
        self.buffer.clear()
        ep_rewards = []
        current_ep_reward = 0.0

        for _ in range(self.n_steps):
            action_np, log_prob, value = self.select_action(self.obs)

            next_obs, reward, terminated, truncated, _ = env.step(int(action_np))
            done = terminated or truncated

            self.buffer.store(self.obs, action_np, reward, done, log_prob, value)
            current_ep_reward += reward
            self.obs = next_obs

            if done:
                ep_rewards.append(current_ep_reward)
                current_ep_reward = 0.0
                self.obs, _ = env.reset()

        # Bootstrap value for GAE
        state_t = torch.FloatTensor(self.obs).unsqueeze(0).to(device)
        with torch.no_grad():
            _, _, _, last_value = self.network.get_action(state_t)
        self.buffer.compute_returns_and_advantages(last_value.item(), self.gamma, self.gae_lambda)

        return np.mean(ep_rewards) if ep_rewards else 0.0

    def update(self):
        """
        Run n_epochs of PPO updates on the collected rollout.

        Returns:
            Tuple of (mean_policy_loss, mean_value_loss, mean_entropy)
        """
        all_policy_losses, all_value_losses, all_entropies = [], [], []

        for _ in range(self.n_epochs):
            for batch in self.buffer.get_batches(self.batch_size):
                states, actions, old_log_probs, advantages, returns, old_values = batch
                states = states.to(device)
                actions = actions.to(device)
                old_log_probs = old_log_probs.to(device)
                advantages = advantages.to(device)
                returns = returns.to(device)
                old_values = old_values.to(device)

                # Normalize advantages per mini-batch (standard PPO practice)
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

                # Re-evaluate actions under current policy
                _, new_log_probs, entropy, new_values = self.network.get_action(states, actions)
                new_values = new_values.squeeze(-1)

                # PPO clipped surrogate objective
                log_ratio = new_log_probs - old_log_probs
                ratio = log_ratio.exp()

                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss with clipping — prevents V from drifting too far per epoch
                # (standard in CleanRL; keeps multi-step advantages valid across epochs)
                v_loss_unclip = F.mse_loss(new_values, returns)
                v_clipped = old_values + torch.clamp(
                    new_values - old_values, -self.clip_eps, self.clip_eps
                )
                v_loss_clipped = F.mse_loss(v_clipped, returns)
                value_loss = torch.max(v_loss_unclip, v_loss_clipped)

                entropy_loss = -entropy.mean()

                loss = policy_loss + self.value_coef * value_loss + self.entropy_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.network.parameters(), self.max_grad_norm)
                self.optimizer.step()

                all_policy_losses.append(policy_loss.item())
                all_value_losses.append(value_loss.item())
                all_entropies.append(entropy.mean().item())

        return np.mean(all_policy_losses), np.mean(all_value_losses), np.mean(all_entropies)

print('PPOAgent defined (SOLUTION)')

## Training Functions — Complete Solution

In [ ]:
def train_reinforce(env_name='CartPole-v1', num_episodes=600, lr=1e-3,
                    gamma=0.99, use_baseline=True, value_coef=0.5,
                    seed=42, solve_threshold=None):
    """
    Train a REINFORCE agent.

    Args:
        env_name        (str):   Gymnasium environment name.
        num_episodes    (int):   Maximum number of training episodes.
        lr              (float): Learning rate.
        gamma           (float): Discount factor.
        use_baseline    (bool):  Whether to use the value baseline.
        value_coef      (float): Coefficient for value loss.
        seed            (int):   Random seed.
        solve_threshold (float): Stop early when avg reward (last 100 eps) >= this.

    Returns:
        dict with keys: episode_rewards, pg_losses, value_losses, agent
    """
    set_seed(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = REINFORCEAgent(state_dim, action_dim, lr=lr, gamma=gamma,
                           use_baseline=use_baseline, value_coef=value_coef, seed=seed)

    episode_rewards, pg_losses, value_losses = [], [], []

    for episode in range(num_episodes):
        obs, _ = env.reset(seed=seed + episode)
        states, actions, rewards = [], [], []
        episode_reward = 0.0

        for _ in range(10000):
            action, _ = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            states.append(obs)
            actions.append(action)
            rewards.append(reward)
            episode_reward += reward
            obs = next_obs
            if terminated or truncated:
                break

        pg_loss, val_loss = agent.update(states, actions, rewards)
        episode_rewards.append(episode_reward)
        pg_losses.append(pg_loss)
        value_losses.append(val_loss)

        if (episode + 1) % 50 == 0:
            avg = np.mean(episode_rewards[-50:])
            baseline_str = 'with baseline' if use_baseline else 'no baseline'
            print(f'Episode {episode+1}/{num_episodes} | Avg Reward (50): {avg:.1f} | '
                  f'PG Loss: {pg_loss:.4f} | {baseline_str}')

        if solve_threshold is not None and len(episode_rewards) >= 100:
            if np.mean(episode_rewards[-100:]) >= solve_threshold:
                print(f'Solved at episode {episode+1}! '
                      f'Avg reward: {np.mean(episode_rewards[-100:]):.1f}')
                break

    env.close()
    return {'episode_rewards': episode_rewards, 'pg_losses': pg_losses,
            'value_losses': value_losses, 'agent': agent}


def evaluate_ppo(agent, env, num_episodes=5, seed=0):
    """Evaluate a PPO agent (discrete actions)."""
    rewards = []
    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_reward = 0.0
        for _ in range(10000):
            state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            with torch.no_grad():
                action, _, _, _ = agent.network.get_action(state_t)
            obs, reward, terminated, truncated, _ = env.step(int(action.item()))
            ep_reward += reward
            if terminated or truncated:
                break
        rewards.append(ep_reward)
    return np.mean(rewards)


def train_ppo(env_name='CartPole-v1', num_updates=200, n_steps=512,
              seed=42, solve_threshold=None, anneal_lr=True, **agent_kwargs):
    """
    Train a PPO agent with optional linear LR annealing (CleanRL-style).

    The env state is carried between collect_rollout calls (agent.obs) so
    rollouts start where the previous one left off — matching CleanRL behavior.

    Args:
        env_name        (str):  Gymnasium environment name.
        num_updates     (int):  Number of rollout-update cycles.
        n_steps         (int):  Environment steps per rollout.
        seed            (int):  Random seed.
        solve_threshold (float):Stop early when avg eval reward >= this.
        anneal_lr       (bool): Linearly decay LR from initial value to 0.
        **agent_kwargs:         Passed to PPOAgent (lr, gae_lambda, clip_eps, etc.)

    Returns:
        dict with keys: update_rewards, policy_losses, value_losses, entropies, agent
    """
    set_seed(seed)
    env = gym.make(env_name)
    eval_env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = PPOAgent(state_dim, action_dim, n_steps=n_steps, seed=seed, **agent_kwargs)

    # Initialize env state — carried between rollouts (CleanRL-style)
    agent.obs, _ = env.reset(seed=seed)

    # Linear LR annealing scheduler: lr * (1 - update/num_updates)
    if anneal_lr:
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            agent.optimizer,
            lr_lambda=lambda step: 1.0 - step / num_updates
        )

    update_rewards, policy_losses, value_losses, entropies = [], [], [], []

    for update in range(num_updates):
        rollout_reward = agent.collect_rollout(env)
        pg_loss, val_loss, entropy = agent.update()

        if anneal_lr:
            scheduler.step()

        eval_reward = evaluate_ppo(agent, eval_env, num_episodes=5, seed=seed)
        update_rewards.append(eval_reward)
        policy_losses.append(pg_loss)
        value_losses.append(val_loss)
        entropies.append(entropy)

        if (update + 1) % 20 == 0:
            current_lr = agent.optimizer.param_groups[0]['lr']
            avg = np.mean(update_rewards[-20:])
            print(f'Update {update+1}/{num_updates} | '
                  f'Avg Eval Reward (20): {avg:.1f} | '
                  f'PG Loss: {pg_loss:.4f} | '
                  f'Entropy: {entropy:.4f} | '
                  f'LR: {current_lr:.2e}')

        if solve_threshold is not None and len(update_rewards) >= 10:
            if np.mean(update_rewards[-10:]) >= solve_threshold:
                print(f'Solved at update {update+1}!')
                break

    env.close()
    eval_env.close()
    return {'update_rewards': update_rewards, 'policy_losses': policy_losses,
            'value_losses': value_losses, 'entropies': entropies, 'agent': agent}

print('Training functions defined (SOLUTION)')

## Plotting Utilities

In [121]:
def _save_plot(fig, title):
    try:
        plots_dir = os.path.join(EXPERIMENTS_DIR, 'plots')
        os.makedirs(plots_dir, exist_ok=True)
        fname = title.lower().replace(' ', '_').replace('/', '_').replace('—', '').replace(':', '').strip('_') + '.png'
        fig.savefig(os.path.join(plots_dir, fname), dpi=150, bbox_inches='tight')
        print(f'Plot saved: {os.path.join(plots_dir, fname)}')
    except Exception as e:
        print(f'Could not save plot: {e}')


def plot_reinforce_results(results, title='REINFORCE Training', window=50):
    """Plot REINFORCE training curves (episode-based)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    rewards = results['episode_rewards']

    axes[0].plot(rewards, alpha=0.3, color='blue', label='Raw')
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
        axes[0].plot(range(window - 1, len(rewards)), ma, color='blue', label=f'{window}-ep avg')
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
    axes[0].set_title(f'{title} — Episode Rewards'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(results['pg_losses'], alpha=0.5, color='red')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Policy Loss')
    axes[1].set_title(f'{title} — Policy Gradient Loss'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(results['value_losses'], alpha=0.5, color='green')
    axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Value Loss')
    axes[2].set_title(f'{title} — Value Loss'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    _save_plot(fig, title)
    plt.show()


def plot_ppo_results(results, title='PPO Training', window=20):
    """Plot PPO training curves (update-based)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    rewards = results['update_rewards']

    axes[0].plot(rewards, alpha=0.4, color='blue', label='Eval reward')
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
        axes[0].plot(range(window - 1, len(rewards)), ma, color='blue', label=f'{window}-update avg')
    axes[0].set_xlabel('PPO Update'); axes[0].set_ylabel('Reward')
    axes[0].set_title(f'{title} — Eval Reward'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(results['policy_losses'], alpha=0.5, color='red')
    axes[1].set_xlabel('PPO Update'); axes[1].set_ylabel('Policy Loss')
    axes[1].set_title(f'{title} — Policy Loss'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(results['entropies'], alpha=0.5, color='purple')
    axes[2].set_xlabel('PPO Update'); axes[2].set_ylabel('Entropy')
    axes[2].set_title(f'{title} — Policy Entropy'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    _save_plot(fig, title)
    plt.show()


def plot_comparison(all_results, title='Comparison', window=50, x_label='Episode', reward_key='episode_rewards'):
    """Plot reward comparison across methods/seeds (mean ± std)."""
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['blue', 'red', 'green', 'orange', 'purple']

    for idx, (name, results_list) in enumerate(all_results.items()):
        color = colors[idx % len(colors)]
        min_len = min(len(r[reward_key]) for r in results_list)
        all_rewards = np.array([r[reward_key][:min_len] for r in results_list])

        if min_len >= window:
            smoothed = np.array([np.convolve(row, np.ones(window) / window, mode='valid')
                                 for row in all_rewards])
            mean, std = smoothed.mean(axis=0), smoothed.std(axis=0)
            x = range(window - 1, min_len)
        else:
            mean, std = all_rewards.mean(axis=0), all_rewards.std(axis=0)
            x = range(min_len)

        ax.plot(x, mean, color=color, label=name)
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.15)

    ax.set_xlabel(x_label); ax.set_ylabel('Reward'); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    _save_plot(fig, title)
    plt.show()

print('Plotting utilities defined!')

Plotting utilities defined!


## Train REINFORCE on CartPole-v1

In [ ]:
set_seed(42)
results_rf = train_reinforce(
    env_name='CartPole-v1',
    num_episodes=2000,
    lr=1e-3,
    use_baseline=True,
    solve_threshold=475,
    seed=42,
)

plot_reinforce_results(results_rf, title='REINFORCE on CartPole-v1')
avg = np.mean(results_rf['episode_rewards'][-100:])
print(f'\nAverage reward (last 100 eps): {avg:.1f}')
print('CartPole SOLVED!' if avg >= 475 else 'Not yet solved.')

Episode 50/2000 | Avg Reward (50): 26.8 | PG Loss: -0.0120 | with baseline
Episode 100/2000 | Avg Reward (50): 25.1 | PG Loss: -0.0098 | with baseline
Episode 150/2000 | Avg Reward (50): 33.0 | PG Loss: 0.0228 | with baseline


## Visualise — REINFORCE Agent on CartPole-v1

In [ ]:
def record_video(agent, env_name, video_dir, episode_trigger=lambda ep: ep == 0, seed=42):
    """Record one episode of the agent and return the video path."""
    os.makedirs(video_dir, exist_ok=True)
    env = gym.make(env_name, render_mode='rgb_array')
    env = gym.wrappers.RecordVideo(env, video_folder=video_dir,
                                   episode_trigger=episode_trigger,
                                   disable_logger=True)
    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0
    for _ in range(10000):
        state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        with torch.no_grad():
            action, _, _, _ = agent.network.get_action(state_t)
        action_np = action.cpu().numpy().squeeze(0)
        env_action = int(action_np)
        obs, reward, terminated, truncated, _ = env.step(env_action)
        ep_reward += reward
        if terminated or truncated:
            break
    env.close()
    print(f'Episode reward: {ep_reward:.1f}')
    videos = sorted([f for f in os.listdir(video_dir) if f.endswith('.mp4')])
    return os.path.join(video_dir, videos[-1]) if videos else None


video_dir = os.path.join(EXPERIMENTS_DIR, 'videos', 'reinforce_cartpole')
video_path = record_video(results_rf['agent'], 'CartPole-v1', video_dir, seed=42)

if video_path:
    display(Video(video_path, embed=True, width=400))
else:
    print('No video found — check that ffmpeg is installed (sudo apt install ffmpeg).')

## Train PPO on CartPole-v1

In [ ]:
set_seed(42)
results_ppo_cp = train_ppo(
    env_name='CartPole-v1',
    num_updates=1000,
    n_steps=2048,
    solve_threshold=475,
    seed=42,
    lr=3e-4,
    gae_lambda=0.95,
    clip_eps=0.2,
    n_epochs=4,
    batch_size=64,
    entropy_coef=0.01,
)

plot_ppo_results(results_ppo_cp, title='PPO on CartPole-v1')
avg_ppo = np.mean(results_ppo_cp['update_rewards'][-10:])
solved  = avg_ppo >= 475 or len(results_ppo_cp['update_rewards']) < 1000
print(f'\nAverage eval reward (last 10 updates): {avg_ppo:.1f}')
print('CartPole SOLVED!' if solved else 'Not yet solved.')

## Visualise — PPO Agent on CartPole-v1

In [ ]:
video_dir = os.path.join(EXPERIMENTS_DIR, 'videos', 'ppo_cartpole')
video_path = record_video(results_ppo_cp['agent'], 'CartPole-v1', video_dir, seed=42)

if video_path:
    display(Video(video_path, embed=True, width=400))
else:
    print('No video found — check that ffmpeg is installed (sudo apt install ffmpeg).')

---

# Part B — Experiments

---

**For all experiments:** Run at least **3 different seeds** and report mean ± std.

## B.1 — REINFORCE vs PPO on CartPole-v1

Compare the two algorithms on convergence speed, variance, and final performance.

In [ ]:
seeds = [42, 123, 456]

rf_results = [
    train_reinforce(env_name='CartPole-v1', num_episodes=2000, use_baseline=True, seed=s)
    for s in seeds
]

# n_epochs=4 (CleanRL standard) keeps advantages fresh across epochs.
# n_epochs=10 causes V to shift too much per update, making multi-step GAE
# advantages stale — which artificially penalizes lambda=0.95 vs lambda=0.
ppo_results = [
    train_ppo(env_name='CartPole-v1', num_updates=500, n_steps=2048, seed=s,
              lr=3e-4, gae_lambda=0.95, clip_eps=0.2, n_epochs=4,
              batch_size=64, entropy_coef=0.01, solve_threshold=475)
    for s in seeds
]

# ── Common-axis plot: environment steps ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
window = 20

# REINFORCE — x = cumulative env steps (reward = steps per episode in CartPole)
rf_color = 'steelblue'
rf_min = min(len(r['episode_rewards']) for r in rf_results)
rf_all = np.array([r['episode_rewards'][:rf_min] for r in rf_results])
rf_smoothed = np.array([np.convolve(row, np.ones(window)/window, mode='valid') for row in rf_all])
cum_steps_rf = np.cumsum(rf_all[0])[window - 1:]
ax.plot(cum_steps_rf, rf_smoothed.mean(0), color=rf_color, label='REINFORCE (with baseline)', linewidth=2)
ax.fill_between(cum_steps_rf,
                rf_smoothed.mean(0) - rf_smoothed.std(0),
                rf_smoothed.mean(0) + rf_smoothed.std(0),
                color=rf_color, alpha=0.15)

# PPO — x = update * n_steps
# Pad solved runs with their final reward so all curves extend to the same length
ppo_color = 'darkorange'
ppo_max = max(len(r['update_rewards']) for r in ppo_results)
ppo_padded = [
    r['update_rewards'] + [r['update_rewards'][-1]] * (ppo_max - len(r['update_rewards']))
    for r in ppo_results
]
ppo_all = np.array(ppo_padded)
ppo_steps = np.arange(1, ppo_max + 1) * 2048
ppo_smoothed = np.array([np.convolve(row, np.ones(window)/window, mode='valid') for row in ppo_all])
ppo_steps_smooth = ppo_steps[window - 1:]
ax.plot(ppo_steps_smooth, ppo_smoothed.mean(0), color=ppo_color, label='PPO', linewidth=2)
ax.fill_between(ppo_steps_smooth,
                ppo_smoothed.mean(0) - ppo_smoothed.std(0),
                ppo_smoothed.mean(0) + ppo_smoothed.std(0),
                color=ppo_color, alpha=0.15)

ax.axhline(475, color='gray', linestyle='--', linewidth=1, label='Solve threshold (475)')
ax.set_xlabel('Environment steps'); ax.set_ylabel('Reward')
ax.set_title('REINFORCE vs PPO on CartPole-v1 (3 seeds, common x-axis: env steps)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
_save_plot(fig, 'REINFORCE vs PPO — env steps')
plt.show()

# ── Summary stats ─────────────────────────────────────────────────────────────
rf_final   = [np.mean(r['episode_rewards'][-100:]) for r in rf_results]
ppo_final  = [np.mean(r['update_rewards'][-20:])   for r in ppo_results]
rf_solved  = [len(r['episode_rewards'])  for r in rf_results]
ppo_solved = [len(r['update_rewards'])   for r in ppo_results]
rf_steps   = [int(np.sum(r['episode_rewards'])) for r in rf_results]
ppo_steps_s = [n * 2048 for n in ppo_solved]
print(f'REINFORCE final avg reward: {np.mean(rf_final):.1f} ± {np.std(rf_final):.1f}')
print(f'PPO final avg reward:       {np.mean(ppo_final):.1f} ± {np.std(ppo_final):.1f}')
print(f'\nREINFORCE: {rf_solved} episodes  (~{[f"{s//1000}k" for s in rf_steps]} env steps)')
print(f'PPO:       {ppo_solved} updates   (~{[f"{s//1000}k" for s in ppo_steps_s]} env steps)')

## B.2 — Ablation: Baseline in REINFORCE

Does the value baseline actually reduce variance and speed up learning?

In [ ]:
seeds = [42, 123, 456]

baseline_results = {
    'REINFORCE (with baseline)': [
        train_reinforce(env_name='CartPole-v1', num_episodes=2000, use_baseline=True, seed=s)
        for s in seeds
    ],
    'REINFORCE (no baseline)': [
        train_reinforce(env_name='CartPole-v1', num_episodes=2000, use_baseline=False, seed=s)
        for s in seeds
    ],
}

plot_comparison(baseline_results, title='REINFORCE: Baseline Ablation on CartPole-v1',
                window=50, x_label='Episode', reward_key='episode_rewards')

for name, results_list in baseline_results.items():
    final = [np.mean(r['episode_rewards'][-100:]) for r in results_list]
    print(f'{name}: {np.mean(final):.1f} ± {np.std(final):.1f}')

## B.3 — Ablation: GAE Lambda

GAE interpolates between one-step TD ($\lambda = 0$, low variance, high bias) and Monte Carlo ($\lambda = 1$, high variance, zero bias). This ablation shows how $\lambda$ affects convergence speed and stability on CartPole.

In [ ]:
seeds = [42, 123, 456]
gae_lambdas = [0.0, 0.95, 1.0]  # TD (high bias), default, MC (no bias)

# No solve_threshold here — we want to see the full learning curve for each lambda.
# n_epochs=4 (standard) is important: with n_epochs=10, V shifts too much per update,
# making lambda=0 (pure TD, uses current V) look better than lambda=0.95 artificially.
gae_results = {}
for lam in gae_lambdas:
    gae_results[f'lambda={lam}'] = [
        train_ppo(env_name='CartPole-v1', num_updates=600, n_steps=2048, seed=s,
                  gae_lambda=lam, lr=3e-4, clip_eps=0.2, n_epochs=4,
                  batch_size=64, entropy_coef=0.01)
        for s in seeds
    ]

plot_comparison(gae_results, title='PPO: Effect of GAE Lambda on CartPole-v1',
                window=20, x_label='PPO Update', reward_key='update_rewards')

for name, results_list in gae_results.items():
    final = [np.mean(r['update_rewards'][-20:]) for r in results_list]
    print(f'{name}: {np.mean(final):.1f} ± {np.std(final):.1f}')

---

**Lab designed by Amath Sow:** amath.sow@liu.se

**TDDE78 — Deep Reinforcement Learning, Linköping University, Spring 2026**

---

## Summary

### B.1 — REINFORCE vs PPO

*[Update with actual numbers after re-running with the fixed implementation.]*

**Expected behavior:** PPO reaches the solve threshold (~475) significantly faster than REINFORCE in env-step terms. With n_steps=2048 each PPO update covers 2048 env steps; PPO typically solves in ~100–200 updates (~200k–400k steps). REINFORCE typically needs 800–2000 episodes on CartPole, corresponding to ~300k–700k env steps, with high inter-seed variance.

**Key insight:** PPO's off-policy reuse (multiple epochs on the same rollout) makes it far more sample efficient. REINFORCE uses each episode exactly once and discards it. The PPO curve should be visibly faster and show lower inter-seed variance once all seeds converge.

---

### B.2 — Baseline Ablation

*[Update with actual numbers after re-running.]*

**Expected behavior:** REINFORCE with and without baseline achieve similar final performance on CartPole (~495). The baseline may or may not speed convergence on this specific environment:
- CartPole returns are near-deterministic once the policy improves → little variance to reduce
- An inaccurate V early in training adds noise to the advantage signal
- The baseline is more beneficial on harder tasks with highly variable returns across states

---

### B.3 — GAE Lambda Ablation

*[Update with actual numbers after re-running with n_epochs=4.]*

**Expected behavior with n_epochs=4 (standard PPO):**
- **λ = 0.0 (pure TD):** Slowest and most unstable — advantages depend entirely on V being accurate, which it is not at initialization. High bootstrap bias.
- **λ = 0.95 (default):** Best or near-best. Good bias-variance tradeoff. Standard choice across environments.
- **λ = 1.0 (Monte Carlo):** Fast on CartPole specifically — short episodes (≤500 steps) keep MC variance manageable, and zero bootstrap bias helps early in training.

**Why n_epochs matters:** With n_epochs=10, V shifts significantly within each update cycle. Lambda=0 (pure TD, uses the freshest V) then appears better than lambda=0.95 (depends on the V from rollout time staying valid). Reducing to n_epochs=4 keeps advantages valid across epochs, restoring the expected ranking.

**Key insight:** λ = 0.95 is the standard safe default; λ = 1.0 can win on short-episode tasks. The best λ is environment-dependent.